## 1. Imports et paramètres

Chargement des bibliothèques et définition des constantes du projet

In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

GTFS_DIR   = Path("../raw/atoumod-gtfs_20260512")
OUTPUT     = Path("../data/arrets_2026.gpkg")
LAYER_NAME = "arrets_2026"

## 2. Lecture des fichiers GTFS

Chargement des quatre tables nécessaires depuis le dossier GTFS ATOUMOD :
`routes.txt`, `trips.txt`, `stop_times.txt`, `stops.txt`.
La jointure entre ces tables suit la chaîne : `stops → stop_times → trips → routes`.

In [ ]:
routes     = pd.read_csv(GTFS_DIR / "routes.txt",
                         usecols=["route_id", "route_short_name", "route_type"])
trips      = pd.read_csv(GTFS_DIR / "trips.txt",
                         usecols=["trip_id", "route_id"])
stop_times = pd.read_csv(GTFS_DIR / "stop_times.txt",
                         usecols=["trip_id", "stop_id"], low_memory=False)
stops      = pd.read_csv(GTFS_DIR / "stops.txt")

print(f"routes     : {len(routes):>6} lignes")
print(f"trips      : {len(trips):>6} lignes")
print(f"stop_times : {len(stop_times):>6} lignes")
print(f"stops      : {len(stops):>6} lignes")

## 3. Périmètre de la Métropole Rouen Normandie

Téléchargement du périmètre administratif de la MRN via OSMnx / Nominatim.
Utilisé pour filtrer les arrêts par localisation réelle plutôt que par
département (FR:76), ce qui exclut les opérateurs desservant le 76
hors MRN (ex. ATOUMOD005 — Dieppe Mob).

**Source** : OpenStreetMap via API Nominatim  
**Projection** : EPSG:2154  
**Sortie** : `data/perimetre_MRN.gpkg`

In [ ]:
import osmnx as ox

mrn = ox.geocode_to_gdf("Métropole Rouen Normandie, France")
mrn = mrn.to_crs("EPSG:2154")

perimetre_path = Path("../data/perimetre_MRN.gpkg")
mrn.to_file(perimetre_path, driver="GPKG")
print(f"✓ Périmètre exporté : {perimetre_path}")
print(f"  Superficie : {mrn.geometry.area.sum()/1e6:.0f} km²")
print(f"  CRS : {mrn.crs}")

## 4. Filtre spatial des arrêts dans la MRN


In [ ]:
# Tous les stops → GeoDataFrame
stops["operateur"] = stops["stop_id"].str.split(":").str[-1]
stops_gdf = gpd.GeoDataFrame(
    stops,
    geometry=gpd.points_from_xy(stops["stop_lon"], stops["stop_lat"]),
    crs="EPSG:4326"
).to_crs("EPSG:2154")

# Filtre spatial : arrêts dans la MRN
stops_mrn = stops_gdf[stops_gdf.within(mrn.union_all())]
print(f"Arrêts dans le périmètre MRN : {len(stops_mrn)}")

## 5. Jointure et attribution du mode

Jointure vectorisée `stops → stop_times → trips → routes` sur les arrêts MRN.
Le mode est attribué par `np.select` sur `route_type` et `route_short_name`.

| Critère                                                | Mode  |
|--------------------------------------------------------|-------|
| `route_type = 1`                                       | Tram  |
| `route_type = 2`                                       | Train |
| `route_type = 3` et `route_short_name` dans TEOR_LINES | TEOR  |
| `route_type = 3` autres                                | Bus   |
| `route_type = 4`                                       | Bac   |

Résultat : un index `(stop_id × mode)` avec la liste des lignes
desservant chaque arrêt pour chaque mode.

In [ ]:
import numpy as np

TEOR_LINES = {"T1", "T2", "T3", "T4", "T5"}

# Attribution du mode par ligne (vectorisé)
conditions = [
    routes["route_type"] == 1,
    routes["route_type"] == 2,
    (routes["route_type"] == 3) & (routes["route_short_name"].isin(TEOR_LINES)),
    (routes["route_type"] == 3) & (~routes["route_short_name"].isin(TEOR_LINES)),
    routes["route_type"] == 4,
]
choix = ["Tram", "Train", "TEOR", "Bus", "Bac"]
routes["mode"] = np.select(conditions, choix, default="Inconnu")

# Jointure stop_times → trips → routes
merged = (stop_times
    .merge(trips, on="trip_id")
    .merge(routes[["route_id", "route_short_name", "mode"]], on="route_id")
)

# Index (stop_id × mode) → liste des lignes, restreint aux arrêts MRN
idx = (merged[merged["stop_id"].isin(stops_mrn["stop_id"])]
    [["stop_id", "mode", "route_short_name"]]
    .drop_duplicates()
    .groupby(["stop_id", "mode"])["route_short_name"]
    .agg(lambda x: ", ".join(sorted(x)))
    .reset_index()
    .rename(columns={"route_short_name": "lignes"})
)
print(f"Combinaisons (arrêt × mode) dans la MRN : {len(idx)}")

## 6. Arrêts sans lignes

Identification des arrêts MRN absents de `stop_times` — leur mode
ne peut pas être déterminé par la jointure GTFS.

In [ ]:
agency = pd.read_csv(GTFS_DIR / "agency.txt")
agency["code"] = agency["agency_id"].str.split(":").str[0]

sans_passage = stops_mrn[~stops_mrn["stop_id"].isin(idx["stop_id"])]

sans_df = (sans_passage
    .groupby("operateur")
    .size()
    .reset_index(name="n_arrets")
    .merge(agency[["code", "agency_name"]],
           left_on="operateur", right_on="code", how="left")
    .sort_values("n_arrets", ascending=False)
)
print(f"Arrêts sans passage dans stop_times : {len(sans_passage)}")
print(sans_df[["operateur", "agency_name", "n_arrets"]].to_string(index=False))

## 7. Examen des arrêts sans passage

Les 496 arrêts sans passage dans `stop_times` appartiennent tous à
ATOUMOD001 (Astuce).
On les visualise (en rouge) avec les arrêts Tram (en bleu) et TEOR (en jaune).

In [ ]:
import folium

sans_wgs = sans_passage.to_crs("EPSG:4326")
tram_wgs = gpd.GeoDataFrame(
    idx[idx["mode"] == "Tram"].merge(stops_mrn, on="stop_id"),
    geometry="geometry", crs="EPSG:2154"
).to_crs("EPSG:4326")
teor_wgs = gpd.GeoDataFrame(
    idx[idx["mode"] == "TEOR"].merge(stops_mrn, on="stop_id"),
    geometry="geometry", crs="EPSG:2154"
).to_crs("EPSG:4326")

m_sans = folium.Map(
    location=[49.44, 1.09],
    zoom_start=11,
    tiles="CartoDB positron"
)

# Arrêts sans passage (rouge)
fg_sans = folium.FeatureGroup(name="Sans passage (496)")
for _, r in sans_wgs.iterrows():
    folium.CircleMarker(
        location=[r.geometry.y, r.geometry.x],
        radius=4,
        color="#e74c3c",
        fill=True,
        fill_opacity=0.8,
        tooltip=f"{r['stop_name']} — {r['stop_id']}",
    ).add_to(fg_sans)
fg_sans.add_to(m_sans)

# Arrêts Tram (jaune)
fg_tram = folium.FeatureGroup(name="Tram")
for _, r in tram_wgs.iterrows():
    folium.CircleMarker(
        location=[r.geometry.y, r.geometry.x],
        radius=4,
        color="#1f77b4",
        fill=True,
        fill_opacity=0.8,
        tooltip=f"{r['stop_name']} — Tram — {r['lignes']}",
    ).add_to(fg_tram)
fg_tram.add_to(m_sans)

# Arrêts TEOR (bleu)
fg_teor = folium.FeatureGroup(name="TEOR")
for _, r in teor_wgs.iterrows():
    folium.CircleMarker(
        location=[r.geometry.y, r.geometry.x],
        radius=4,
        color="#f1c40f",
        fill=True,
        fill_opacity=0.8,
        tooltip=f"{r['stop_name']} — TEOR — {r['lignes']}",
    ).add_to(fg_teor)
fg_teor.add_to(m_sans)

folium.LayerControl().add_to(m_sans)
m_sans.save("../data/arrets_sans_passage.html")
print(f"✓ Carte sauvegardée ({len(sans_wgs)} arrêts sans passage, "
      f"{len(tram_wgs)} Tram, {len(teor_wgs)} TEOR)")

## 8. Traitement des arrêts sans passage

Les 496 arrêts sans passage dans `stop_times` appartiennent tous à
ATOUMOD001 (Astuce) et sont distincts des arrêts Tram et TEOR d'Astuce qui sont tous
présents dans `stop_times`.
Ces 496 arrêts reçoivent donc le mode `Bus`.
Le GeoDataFrame peut être généré.

In [ ]:
rows = []

# Arrêts avec passages — une ligne par (stop_id × mode)
avec = stops_mrn.merge(idx, on="stop_id", how="inner")
for _, r in avec.iterrows():
    rows.append({
        "nom":       r["stop_name"],
        "horizon":   "2026",
        "mode":      r["mode"],
        "lignes":    r["lignes"],
        "operateur": r["operateur"],
        "id_source": r["stop_id"],
        "geometry":  r["geometry"],
    })

# Arrêts sans passage — mode Bus (tous ATOUMOD001/Astuce, cf. cellule 6)
for _, r in sans_passage.iterrows():
    rows.append({
        "nom":       r["stop_name"],
        "horizon":   "2026",
        "mode":      "Bus",
        "lignes":    "",
        "operateur": r["operateur"],
        "id_source": r["stop_id"],
        "geometry":  r["geometry"],
    })

gdf = gpd.GeoDataFrame(rows, crs="EPSG:2154")
print(f"Lignes totales (arrêts × modes) : {len(gdf)}")
print(f"  dont arrêts avec passages     : {len(avec)}")
print(f"  dont arrêts sans passage      : {len(sans_passage)}")

## 9. Contrôle qualité

Vérification de la répartition des arrêts par mode et par opérateur.

In [ ]:
# Contrôle rapide à ajouter en cellule 5
print("Répartition des arrêts par mode :")
display(gdf["mode"].value_counts().to_frame())

print("\nArrêts Tram :")
display(gdf[gdf["mode"] == "Tram"][["nom", "lignes", "operateur"]].head(10))

print("\nArrêts TEOR :")
display(gdf[gdf["mode"] == "TEOR"][["nom", "lignes", "operateur"]].head(10))

print("\nArrêts sans lignes :")
display(gdf[gdf["lignes"] == ""][["nom", "mode", "operateur"]].head(10))

print("\nRépartition par opérateur :")
display(gdf["operateur"].value_counts().to_frame())

print(f"\nCRS : {gdf.crs}")
gdf.head(5)

> Les doublons par sens (ex. *Georges Braque ×2*) sont normaux : chaque quai
> directionnel est un `stop_id` distinct dans le GTFS (aller / retour).
> Cette granularité est utile pour les calculs d'isochrones.

## 10. Export

Export de la couche en **GeoPackage** (`arrets_2026.gpkg`, couche `arrets_2026`)
dans le dossier `data/`, aux côtés des graphes OSM piéton et cyclable.

In [ ]:
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
gdf.to_file(OUTPUT, driver="GPKG", layer=LAYER_NAME)
print(f"✓ Exporté : {OUTPUT}  ({len(gdf)} entités)")

La couche `arrets_2026.gpkg` est prête pour les étapes suivantes :

- **Notebook 03** — Calcul des isochrones piéton et cyclable (OSMnx)
- **Notebook 04** — Jointure spatiale logements ↔ isochrones (GeoPandas)
- **Notebook 05** — Comparaison avant / après SERM 2030

## 11. Visualisation de contrôle

Carte interactive Folium des arrêts par mode, centrée sur Rouen (zoom 11).
Chaque couche est activable/désactivable via le contrôle de légende.
La carte est sauvegardée en HTML dans `data/` pour consultation hors Jupyter.

In [ ]:
import folium

# Reprojection en WGS84 pour Folium
gdf_wgs = gdf.to_crs("EPSG:4326")

# Couleurs par mode
couleurs = {
    "Tram":  "#2ca02c",
    "TEOR":  "#1f77b4",
    "Train": "#d62728",
    "Bus":   "#7f7f7f",
    "Bac":   "#17becf",
}

m = folium.Map(
    location=[49.44, 1.09],  # centre Rouen
    zoom_start=11,
    tiles="CartoDB positron"
)

for mode, groupe in gdf_wgs.groupby("mode"):
    fg = folium.FeatureGroup(name=mode)
    for _, r in groupe.iterrows():
        folium.CircleMarker(
            location=[r.geometry.y, r.geometry.x],
            radius=4,
            color=couleurs.get(mode, "#333"),
            fill=True,
            fill_opacity=0.8,
            tooltip=f"{r['nom']} — {r['mode']}" + (f" — {r['lignes']}" if r['lignes'] else ""),
        ).add_to(fg)
    fg.add_to(m)

folium.LayerControl().add_to(m)
m.save("../data/arrets_2026_carte.html")
print("✓ Carte sauvegardée")